# WDC Schema.org Class Analysis: ClaimReview & NewsArticle

**Source:** [Web Data Commons 2024-12 — Schema.org Class-Specific Subsets](https://webdatacommons.org/structureddata/2024-12/stats/schema_org_subsets.html)

## Strategy

**ClaimReview** (56 MB, ~3.9M quads, ~49.7K URLs)  
→ Downloaded directly from the WDC class-specific subset: a single `part_0.gz` file.  
→ Data is **merged JSON-LD + Microdata** — cannot be separated within this subset.

**NewsArticle** (no class-specific subset available)  
→ Retrieved from the WDC **JSON-LD-only** extractor shards, filtering by `rdf:type = schema:NewsArticle`.  
→ Naturally isolated to JSON-LD, using the shard binary-search + streaming approach.

## Analyses

1. **Syntactic / Structural Validity** — namespace correctness, type validity, property validity, domain/range violations  
2. **Semantic Completeness & Richness** — prevalence, property coverage, AvP, characteristic sets

In [ ]:
# !pip install requests pandas lxml tqdm

In [1]:
import requests, gzip, io, re, json, time, threading
from collections import defaultdict, Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from urllib.parse import urlparse
import pandas as pd

# ── WDC endpoints ──────────────────────────────────────────────────────────────
WDC_CLASS_BASE  = "https://data.dws.informatik.uni-mannheim.de/structureddata/2024-12/quads/classspecific"
WDC_JSONLD_LIST = "http://webdatacommons.org/structureddata/2024-12/files/html-embedded-jsonld.list"

# ── Schema.org namespaces ──────────────────────────────────────────────────────
SO_HTTP  = "http://schema.org/"
SO_HTTPS = "https://schema.org/"
RDF_TYPE = "http://www.w3.org/1999/02/22-rdf-syntax-ns#type"

# ── Local cache paths ──────────────────────────────────────────────────────────
DATA_DIR = Path("wdc_class_data")
DATA_DIR.mkdir(exist_ok=True)

CLAIMREVIEW_NQ   = DATA_DIR / "ClaimReview.nq.gz"
NEWSARTICLE_NQ   = DATA_DIR / "NewsArticle_jsonld.nq"

# ── News domains to seed NewsArticle retrieval ─────────────────────────────────
NEWS_DOMAINS = [
    "reuters.com", "apnews.com", "bbc.com", "cnn.com",
    "theguardian.com", "nytimes.com", "aljazeera.com",
    "france24.com", "dw.com", "lemonde.fr",
    "politifact.com", "snopes.com", "factcheck.org", "fullfact.org",
]

print("Setup complete.")
print(f"  ClaimReview  → {CLAIMREVIEW_NQ}")
print(f"  NewsArticle  → {NEWSARTICLE_NQ}")

Setup complete.
  ClaimReview  → wdc_class_data/ClaimReview.nq.gz
  NewsArticle  → wdc_class_data/NewsArticle_jsonld.nq


## Part 1 — ClaimReview: Download Class-Specific Subset

WDC provides a ready-made `ClaimReview/part_0.gz` (~56 MB compressed) containing all quads whose source pages include at least one `schema:ClaimReview` entity. This covers the full corpus — not filtered by domain.

> **Note on format separation:** The class-specific subsets merge JSON-LD and Microdata into a single file. There is no flag per quad to distinguish the extractor. We accept this limitation here; for a pure JSON-LD view use Part 2 (NewsArticle via JSON-LD shards).

In [2]:
def download_class_subset(class_name: str, dest: Path, part: int = 0):
    """Download a WDC class-specific subset part file (keeps it gzipped locally)."""
    url = f"{WDC_CLASS_BASE}/{class_name}/part_{part}.gz"
    if dest.exists():
        print(f"  Already cached: {dest} ({dest.stat().st_size / 1e6:.1f} MB)")
        return
    print(f"  Downloading {url} …")
    with requests.get(url, stream=True, timeout=300) as r:
        r.raise_for_status()
        total = int(r.headers.get("Content-Length", 0))
        done  = 0
        with open(dest, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):  # 1 MB chunks
                f.write(chunk)
                done += len(chunk)
                if total:
                    print(f"\r  {done/1e6:.1f} / {total/1e6:.1f} MB", end="", flush=True)
    print(f"\n  Saved → {dest} ({dest.stat().st_size / 1e6:.1f} MB)")

download_class_subset("ClaimReview", CLAIMREVIEW_NQ)

  3.1 / 57.6 MB

KeyboardInterrupt: 

In [ ]:
# ── N-Quads parser ─────────────────────────────────────────────────────────────
# Format: subject  predicate  object  graph  .
# Subject/predicate/object/graph may be <URI> or _:blank or "literal"...

NQ_RE = re.compile(
    r'^'
    r'(<[^>]*>|_:\S+)'          # subject
    r'\s+'
    r'(<[^>]+>)'                 # predicate (always URI)
    r'\s+'
    r'(<[^>]*>|_:\S+|"(?:[^"\\]|\\.)*"(?:@\w[\w-]*|(?:\^\^<[^>]+>))?)'  # object
    r'\s+'
    r'(<[^>]+>)'                 # graph (named graph = source page URL)
    r'\s+\.\s*$'
)

def parse_nq_line(line: str):
    """Return (subject, predicate, object, graph) strings or None."""
    m = NQ_RE.match(line)
    if not m:
        return None
    return m.group(1), m.group(2), m.group(3), m.group(4)

def strip_angle(s: str) -> str:
    return s[1:-1] if s.startswith("<") and s.endswith(">") else s

def parse_gz_nquads(path: Path):
    """
    Yield (subject, predicate, object, graph) from a .nq.gz file.
    subject/predicate/object/graph are raw strings (angle-bracket URIs, blank nodes, literals).
    """
    with gzip.open(path, "rt", encoding="utf-8", errors="replace") as f:
        for line in f:
            parsed = parse_nq_line(line.rstrip("\n"))
            if parsed:
                yield parsed


# ── Build entity store ─────────────────────────────────────────────────────────
# entity_store[subject] = {
#   "types": set of type URIs,
#   "props": {predicate_URI: [object_str, ...]},
#   "graph": source page URI,
# }

def build_entity_store(nq_path: Path, gz: bool = True):
    """
    Parse an nquads file (optionally gzipped) and build a per-subject entity store.
    Returns: entity_store dict, parse_errors count, total_lines count
    """
    entity_store = defaultdict(lambda: {"types": set(), "props": defaultdict(list), "graph": None})
    parse_errors = 0
    total = 0

    opener = gzip.open(nq_path, "rt", encoding="utf-8", errors="replace") if gz else open(nq_path, encoding="utf-8", errors="replace")

    with opener as f:
        for line in f:
            total += 1
            parsed = parse_nq_line(line.rstrip("\n"))
            if not parsed:
                parse_errors += 1
                continue
            subj, pred, obj, graph = parsed

            es = entity_store[subj]
            if es["graph"] is None:
                es["graph"] = strip_angle(graph)

            pred_uri = strip_angle(pred)
            if pred_uri == RDF_TYPE:
                es["types"].add(strip_angle(obj))
            else:
                es["props"][pred_uri].append(obj)

    return dict(entity_store), parse_errors, total


print("Building ClaimReview entity store …")
cr_store, cr_errors, cr_total = build_entity_store(CLAIMREVIEW_NQ, gz=True)
print(f"  Lines parsed : {cr_total:,}")
print(f"  Parse errors : {cr_errors:,}")
print(f"  Entities     : {len(cr_store):,}")

## Part 2 — NewsArticle: JSON-LD Extractor Shards (Pure JSON-LD)

`schema:NewsArticle` has no WDC class-specific subset. We retrieve it from the **JSON-LD-only** extractor shards:

1. Fetch the shard list (`html-embedded-jsonld.list`)
2. Binary-search to find the shards containing our news domains
3. Two-pass stream: collect all subjects with `rdf:type = schema:NewsArticle`, then collect their full quad sets
4. Save to `NewsArticle_jsonld.nq`

This approach gives **pure JSON-LD data** (no Microdata contamination).

In [ ]:
# ── Fetch shard list ───────────────────────────────────────────────────────────
print("Fetching JSON-LD shard list …")
resp = requests.get(WDC_JSONLD_LIST, timeout=30)
resp.raise_for_status()
shard_urls = [l.strip() for l in resp.text.strip().splitlines() if l.strip()]
print(f"  {len(shard_urls):,} shards total")

# ── Binary-search helpers ──────────────────────────────────────────────────────
_probe_cache: dict[int, str | None] = {}

def get_first_subject(idx: int, retries: int = 3) -> str | None:
    if idx in _probe_cache:
        return _probe_cache[idx]
    url = shard_urls[idx]
    for attempt in range(retries):
        try:
            with requests.get(url, stream=True, timeout=60) as r:
                r.raise_for_status()
                with gzip.GzipFile(fileobj=r.raw) as gz:
                    for raw in gz:
                        line = raw.decode("utf-8", errors="replace").strip()
                        sp   = line.find(" ")
                        if sp < 0:
                            continue
                        subj = line[:sp].strip("<>")
                        if subj.startswith("http"):
                            _probe_cache[idx] = subj
                            return subj
            break
        except Exception:
            time.sleep(2 ** attempt)
    _probe_cache[idx] = None
    return None

def find_shard_lo(target: str, N: int) -> int:
    lo, hi = 0, N - 1
    while lo < hi:
        mid   = (lo + hi) // 2
        first = get_first_subject(mid)
        if first is None or first < target:
            lo = mid + 1
        else:
            hi = mid
    return lo

N = len(shard_urls)
domain_pat = re.compile("|".join(re.escape(d) for d in NEWS_DOMAINS))

# Find shard ranges for each domain
all_shard_indices = set()
for domain in sorted(NEWS_DOMAINS):
    for scheme in ("http://", "https://"):
        prefix = f"{scheme}{domain}/"
        upper  = prefix[:-1] + chr(ord(prefix[-1]) + 1)
        lo = find_shard_lo(prefix, N)
        hi = find_shard_lo(upper,  N)
        for i in range(max(0, lo - 1), min(N, hi + 2)):
            all_shard_indices.add(i)
    print(f"  {domain}")

all_shard_indices = sorted(all_shard_indices)
print(f"\nTotal shards to scan: {len(all_shard_indices)}")

In [ ]:
SO_TYPE_URIS = {
    f"<{SO_HTTP}NewsArticle>",
    f"<{SO_HTTPS}NewsArticle>",
    # also catch subtypes: ReportageNewsArticle, AnalysisNewsArticle, etc.
    f"<{SO_HTTP}Article>",
    f"<{SO_HTTPS}Article>",
}

def stream_shard_for_newsarticle(shard_idx: int):
    """
    Single-pass: collect full lines from news domains with NewsArticle type.
    Returns (matched_lines, total_lines, newsarticle_subjects).
    """
    url = shard_urls[shard_idx]
    lines_by_subj: dict[str, list[str]] = defaultdict(list)
    newsarticle_subjects: set[str] = set()
    total = 0

    try:
        with requests.get(url, stream=True, timeout=300) as r:
            r.raise_for_status()
            with gzip.GzipFile(fileobj=r.raw) as gz:
                for raw in gz:
                    total += 1
                    line = raw.decode("utf-8", errors="replace")
                    # Quick domain filter
                    if not domain_pat.search(line):
                        continue
                    # Store by subject
                    sp = line.find(" ")
                    subj = line[:sp] if sp > 0 else line.strip()
                    lines_by_subj[subj].append(line)
                    # Check for NewsArticle type
                    if RDF_TYPE in line:
                        for t in SO_TYPE_URIS:
                            if t in line:
                                newsarticle_subjects.add(subj)
                                break
    except Exception as e:
        print(f"  ERROR shard {shard_idx}: {e}")
        return [], total, set()

    # Collect all lines for confirmed NewsArticle subjects
    matched = []
    for subj in newsarticle_subjects:
        matched.extend(lines_by_subj[subj])
    return matched, total, newsarticle_subjects

if NEWSARTICLE_NQ.exists():
    print(f"Already cached: {NEWSARTICLE_NQ} ({NEWSARTICLE_NQ.stat().st_size / 1e6:.1f} MB)")
else:
    stats = {"done": 0, "matched": 0, "subjects": 0}
    lock  = threading.Lock()

    open(NEWSARTICLE_NQ, "w").close()
    print(f"Scanning {len(all_shard_indices)} shards with 4 workers …\n")

    with ThreadPoolExecutor(max_workers=4) as executor:
        futures = {executor.submit(stream_shard_for_newsarticle, i): i for i in all_shard_indices}
        for future in as_completed(futures):
            idx = futures[future]
            matched_lines, total_lines, subjects = future.result()
            with lock:
                with open(NEWSARTICLE_NQ, "a", encoding="utf-8") as f:
                    f.writelines(matched_lines)
                stats["done"]     += 1
                stats["matched"]  += len(matched_lines)
                stats["subjects"] += len(subjects)
                print(f"  [{stats['done']:>3}/{len(all_shard_indices)}] shard {idx:05d}: "
                      f"{len(subjects):>5} NewsArticle entities, "
                      f"{len(matched_lines):>7,} lines / {total_lines:>9,} scanned")

    print(f"\nDone. {stats['subjects']:,} NewsArticle entities, "
          f"{stats['matched']:,} lines → {NEWSARTICLE_NQ}")

In [ ]:
print("Building NewsArticle entity store …")
na_store, na_errors, na_total = build_entity_store(NEWSARTICLE_NQ, gz=False)
print(f"  Lines parsed : {na_total:,}")
print(f"  Parse errors : {na_errors:,}")
print(f"  Entities     : {len(na_store):,}")

## Part 3 — Load Schema.org Vocabulary

We fetch the full schema.org vocabulary (JSON-LD format) to:
- Enumerate all valid type names
- Map each type to its valid properties (via `domainIncludes`)
- Map each property to its expected value types (via `rangeIncludes`)

In [ ]:
VOCAB_CACHE = DATA_DIR / "schemaorg_vocab.json"

def load_schemaorg_vocab():
    """
    Download schema.org vocabulary and build:
      all_types  : set of local names (e.g. 'ClaimReview')
      type_props : {type_local_name: set of property local names}
      prop_range : {prop_local_name: set of type/datatype local names}
      type_parents: {type_local_name: set of parent type local names}  (for inheritance)
    """
    if VOCAB_CACHE.exists():
        print("Loading vocab from cache …")
        with open(VOCAB_CACHE) as f:
            raw = json.load(f)
    else:
        url = "https://schema.org/version/latest/schemaorg-current-https.jsonld"
        print(f"Fetching schema.org vocab from {url} …")
        r = requests.get(url, timeout=60)
        r.raise_for_status()
        raw = r.json()
        with open(VOCAB_CACHE, "w") as f:
            json.dump(raw, f)
        print(f"  Cached → {VOCAB_CACHE}")

    graph = raw.get("@graph", [])
    print(f"  {len(graph):,} vocabulary items")

    def local(uri: str) -> str:
        return uri.split("/")[-1].split("#")[-1]

    def ids_of(val) -> list[str]:
        if isinstance(val, dict):
            return [val.get("@id", "")]
        if isinstance(val, list):
            return [v.get("@id", "") if isinstance(v, dict) else str(v) for v in val]
        return [str(val)]

    all_types    = set()
    type_props   = defaultdict(set)   # type → properties defined on it
    prop_range   = defaultdict(set)   # property → expected value types
    type_parents = defaultdict(set)   # type → parent types

    # First pass: collect types and inheritance
    for item in graph:
        item_id   = item.get("@id", "")
        item_type = item.get("@type", "")
        if "rdfs:Class" in str(item_type) or "schema:DataType" in str(item_type):
            name = local(item_id)
            all_types.add(name)
            for parent_id in ids_of(item.get("rdfs:subClassOf", [])):
                if parent_id:
                    type_parents[name].add(local(parent_id))

    # Second pass: collect properties and their domains/ranges
    for item in graph:
        item_id   = item.get("@id", "")
        item_type = item.get("@type", "")
        if "rdf:Property" in str(item_type):
            prop_name = local(item_id)
            for domain_id in ids_of(item.get("schema:domainIncludes", [])):
                if domain_id:
                    type_props[local(domain_id)].add(prop_name)
            for range_id in ids_of(item.get("schema:rangeIncludes", [])):
                if range_id:
                    prop_range[prop_name].add(local(range_id))

    # Expand type_props with inherited properties
    def get_ancestors(t, memo={}):
        if t in memo:
            return memo[t]
        ancestors = set()
        for parent in type_parents.get(t, set()):
            ancestors.add(parent)
            ancestors |= get_ancestors(parent, memo)
        memo[t] = ancestors
        return ancestors

    for t in list(all_types):
        for ancestor in get_ancestors(t):
            type_props[t] |= type_props.get(ancestor, set())

    print(f"  Classes : {len(all_types):,}")
    print(f"  With properties defined: {sum(1 for v in type_props.values() if v):,}")
    return all_types, dict(type_props), dict(prop_range), dict(type_parents)


so_types, so_type_props, so_prop_range, so_type_parents = load_schemaorg_vocab()

# Quick check for our target classes
for cls in ["ClaimReview", "NewsArticle", "Article"]:
    props = so_type_props.get(cls, set())
    print(f"\n  {cls}: {len(props)} properties defined")

In [ ]:
# ── Helper: filter entity store to a target class ─────────────────────────────
def filter_by_class(store: dict, target_class: str) -> dict:
    """Return only entities that have target_class (or a subclass) among their types."""
    result = {}
    for subj, data in store.items():
        for t in data["types"]:
            local_t = t.split("/")[-1].split("#")[-1]
            # Accept exact match or subclass (e.g. ReportageNewsArticle ⊆ NewsArticle)
            if local_t == target_class or target_class in so_type_parents.get(local_t, set()):
                result[subj] = data
                break
    return result

# Extract target-class entities from each store
cr_entities = filter_by_class(cr_store, "ClaimReview")
na_entities = filter_by_class(na_store, "NewsArticle")
# Also look for NewsArticle co-occurrences in the ClaimReview file
na_in_cr    = filter_by_class(cr_store, "NewsArticle")

print(f"ClaimReview entities (from class subset) : {len(cr_entities):,}")
print(f"NewsArticle entities (from JSON-LD shards): {len(na_entities):,}")
print(f"NewsArticle co-occurrences in ClaimReview file: {len(na_in_cr):,}")

# Merge NewsArticle from both sources (deduplicated by subject URI)
na_all = {**na_in_cr, **na_entities}
print(f"NewsArticle combined (deduplicated)      : {len(na_all):,}")

## Part 4 — Syntactic / Structural Validity

### 4.1 Namespace Correctness Rate

Check that schema.org types and properties are referenced with the canonical namespace (`http://schema.org/` or `https://schema.org/`).  
Flag: wrong capitalisation, trailing slashes, typos, or use of other namespaces.

In [ ]:
CANONICAL_NS = (SO_HTTP, SO_HTTPS)

def namespace_report(entities: dict, label: str):
    """
    For each entity, classify every type and property URI:
      - canonical : uses http://schema.org/ or https://schema.org/
      - wrong_ns  : looks like schema.org but with wrong prefix
      - other_ns  : unrelated namespace (rdf:, owl:, etc. — expected and fine)
    """
    canonical = wrong = other = 0
    wrong_examples = Counter()

    for subj, data in entities.items():
        # Check types
        for t in data["types"]:
            if t.startswith(SO_HTTP) or t.startswith(SO_HTTPS):
                canonical += 1
            elif re.search(r'schema\.(org|org/)', t, re.I):
                wrong += 1
                wrong_examples[t] += 1
            else:
                other += 1

        # Check properties
        for prop in data["props"]:
            if prop.startswith(SO_HTTP) or prop.startswith(SO_HTTPS):
                canonical += 1
            elif re.search(r'schema\.(org|org/)', prop, re.I):
                wrong += 1
                wrong_examples[prop] += 1
            else:
                other += 1

    total_schema = canonical + wrong
    rate = canonical / total_schema * 100 if total_schema else float("nan")

    print(f"\n{'─'*60}")
    print(f"  {label}")
    print(f"{'─'*60}")
    print(f"  Canonical schema.org URIs : {canonical:>8,}")
    print(f"  Malformed schema.org URIs : {wrong:>8,}")
    print(f"  Other namespace URIs      : {other:>8,}  (rdf:, owl:, etc.)")
    print(f"  Namespace correctness rate: {rate:.2f}%")
    if wrong_examples:
        print(f"  Malformed examples (top 5):")
        for uri, cnt in wrong_examples.most_common(5):
            print(f"    [{cnt:>5}]  {uri}")
    return rate

cr_ns_rate = namespace_report(cr_entities, "ClaimReview")
na_ns_rate = namespace_report(na_all,      "NewsArticle")

### 4.2 Type Validity Rate

For every `rdf:type` assertion that references schema.org, check whether the local name is a known class in the schema.org vocabulary.

Catches: wrong capitalisation (`newsarticle` vs `NewsArticle`), invented types, deprecated terms.

In [ ]:
def type_validity_report(entities: dict, label: str):
    valid = Counter()
    invalid = Counter()

    for subj, data in entities.items():
        for t in data["types"]:
            if not (t.startswith(SO_HTTP) or t.startswith(SO_HTTPS)):
                continue  # skip rdf:, owl:, etc.
            local_t = t.split("/")[-1].split("#")[-1]
            if local_t in so_types:
                valid[local_t] += 1
            else:
                invalid[local_t] += 1

    total = sum(valid.values()) + sum(invalid.values())
    rate  = sum(valid.values()) / total * 100 if total else float("nan")

    print(f"\n{'─'*60}")
    print(f"  {label}")
    print(f"{'─'*60}")
    print(f"  Valid types   : {sum(valid.values()):>8,}  ({rate:.2f}%)")
    print(f"  Invalid types : {sum(invalid.values()):>8,}  ({100-rate:.2f}%)")
    print(f"\n  Valid type distribution (top 10):")
    for t, c in valid.most_common(10):
        print(f"    {t:<45} {c:>8,}")
    if invalid:
        print(f"\n  Invalid types (top 10):")
        for t, c in invalid.most_common(10):
            print(f"    {t:<45} {c:>8,}")
    return rate, valid, invalid

cr_type_rate, cr_valid_types, cr_invalid_types = type_validity_report(cr_entities, "ClaimReview")
na_type_rate, na_valid_types, na_invalid_types = type_validity_report(na_all,      "NewsArticle")

### 4.3 Property Validity Rate

For each entity, check whether its schema.org properties are actually defined for its type (via `domainIncludes`, with inheritance).

A property is **valid** if it appears in the schema.org vocabulary and `domainIncludes` the entity's type or any of its ancestors.

In [ ]:
def get_all_types_for_entity(data: dict) -> set[str]:
    """Local names of all schema.org types on this entity."""
    result = set()
    for t in data["types"]:
        if t.startswith(SO_HTTP) or t.startswith(SO_HTTPS):
            result.add(t.split("/")[-1].split("#")[-1])
    return result

def prop_validity_report(entities: dict, label: str):
    valid_props   = Counter()
    invalid_props = Counter()

    for subj, data in entities.items():
        entity_types = get_all_types_for_entity(data)
        # Union of all properties allowed for any of the entity's types
        allowed = set()
        for et in entity_types:
            allowed |= so_type_props.get(et, set())

        for prop in data["props"]:
            if not (prop.startswith(SO_HTTP) or prop.startswith(SO_HTTPS)):
                continue  # skip rdf:, owl:, etc.
            prop_local = prop.split("/")[-1].split("#")[-1]
            if prop_local in allowed:
                valid_props[prop_local] += 1
            else:
                invalid_props[prop_local] += 1

    total = sum(valid_props.values()) + sum(invalid_props.values())
    rate  = sum(valid_props.values()) / total * 100 if total else float("nan")

    print(f"\n{'─'*60}")
    print(f"  {label}")
    print(f"{'─'*60}")
    print(f"  Valid property uses   : {sum(valid_props.values()):>8,}  ({rate:.2f}%)")
    print(f"  Invalid property uses : {sum(invalid_props.values()):>8,}  ({100-rate:.2f}%)")
    print(f"\n  Most used valid properties (top 15):")
    for p, c in valid_props.most_common(15):
        print(f"    {p:<45} {c:>8,}")
    if invalid_props:
        print(f"\n  Invalid properties (top 10):")
        for p, c in invalid_props.most_common(10):
            print(f"    {p:<45} {c:>8,}")
    return rate, valid_props, invalid_props

cr_prop_rate, cr_valid_props, cr_invalid_props = prop_validity_report(cr_entities, "ClaimReview")
na_prop_rate, na_valid_props, na_invalid_props = prop_validity_report(na_all,      "NewsArticle")

### 4.4 Domain/Range Violation Rate

Check whether object values match the expected range for their property.

- **Object properties** (range = a schema.org class): value should be a URI (`<…>`) or blank node, not a string literal
- **Date/time properties** (`datePublished`, `dateModified`, etc.): value should match ISO 8601 pattern
- **URL properties** (`url`, `sameAs`): value should be a URI, not a plain literal

In [ ]:
DATE_PROPS = {"datePublished", "dateModified", "dateCreated", "expires",
              "startDate", "endDate", "uploadDate"}
URL_PROPS  = {"url", "sameAs", "mainEntityOfPage", "image", "thumbnail"}

# Datatypes in schema.org that are NOT schema.org classes → expect literals
LITERAL_DATATYPES = {"Text", "Number", "Integer", "Float", "Boolean",
                     "Date", "DateTime", "Time", "URL", "Duration", "Mass",
                     "Energy", "Distance", "Speed", "XPathType", "CssSelectorType"}

ISO_DATE_RE = re.compile(
    r'^\d{4}(-\d{2}(-\d{2}(T\d{2}:\d{2}(:\d{2})?([+-]\d{2}:\d{2}|Z)?)?)?)?$'
)

def is_literal(obj: str) -> bool:
    return obj.startswith('"')

def is_uri_or_blank(obj: str) -> bool:
    return obj.startswith("<") or obj.startswith("_:")

def strip_literal(obj: str) -> str:
    """Extract the string value from a literal like \"foo\"@en or \"foo\"^^<...>."""
    m = re.match(r'^"((?:[^"\\]|\\.)*)"', obj)
    return m.group(1) if m else obj

def range_violation_report(entities: dict, label: str):
    violations = Counter()
    ok         = Counter()
    viol_examples = defaultdict(list)

    for subj, data in entities.items():
        for prop, values in data["props"].items():
            if not (prop.startswith(SO_HTTP) or prop.startswith(SO_HTTPS)):
                continue
            prop_local = prop.split("/")[-1].split("#")[-1]
            ranges     = so_prop_range.get(prop_local, set())

            # Determine expected value kind
            expects_entity  = bool(ranges - LITERAL_DATATYPES) and not ranges.issubset(LITERAL_DATATYPES)
            is_date_prop    = prop_local in DATE_PROPS
            is_url_prop     = prop_local in URL_PROPS

            for obj in values:
                key = prop_local

                if is_date_prop:
                    val = strip_literal(obj)
                    if not ISO_DATE_RE.match(val):
                        violations[key] += 1
                        if len(viol_examples[key]) < 3:
                            viol_examples[key].append(f"date={val!r}")
                    else:
                        ok[key] += 1

                elif is_url_prop:
                    if is_literal(obj) and not strip_literal(obj).startswith("http"):
                        violations[key] += 1
                        if len(viol_examples[key]) < 3:
                            viol_examples[key].append(f"url={strip_literal(obj)[:60]!r}")
                    else:
                        ok[key] += 1

                elif expects_entity:
                    # Should be a URI or blank node, not a plain literal
                    if is_literal(obj):
                        violations[key] += 1
                        if len(viol_examples[key]) < 3:
                            viol_examples[key].append(f"lit={strip_literal(obj)[:60]!r}")
                    else:
                        ok[key] += 1

    total = sum(violations.values()) + sum(ok.values())
    viol_rate = sum(violations.values()) / total * 100 if total else float("nan")

    print(f"\n{'─'*60}")
    print(f"  {label}")
    print(f"{'─'*60}")
    print(f"  Checked value assertions : {total:>8,}")
    print(f"  Range violations         : {sum(violations.values()):>8,}  ({viol_rate:.2f}%)")
    print(f"\n  Top violation properties (top 10):")
    for p, c in violations.most_common(10):
        exs = viol_examples.get(p, [])
        print(f"    {p:<35} {c:>6,}  e.g. {exs}")
    return viol_rate, violations

cr_range_rate, cr_violations = range_violation_report(cr_entities, "ClaimReview")
na_range_rate, na_violations = range_violation_report(na_all,      "NewsArticle")

## Part 5 — Semantic Completeness & Richness

### 5.1 Prevalence & Domain Distribution

How many entities exist per class, and how are they distributed across domains?

In [ ]:
def get_pld(url: str) -> str:
    try:
        host = urlparse(url).netloc or url
        parts = host.split(".")
        return ".".join(parts[-2:]) if len(parts) >= 2 else host
    except Exception:
        return url

def prevalence_report(entities: dict, label: str, top_n: int = 20):
    domain_counts = Counter()
    for subj, data in entities.items():
        graph = data.get("graph") or subj
        domain_counts[get_pld(graph)] += 1

    total = sum(domain_counts.values())
    print(f"\n{'─'*60}")
    print(f"  {label}")
    print(f"{'─'*60}")
    print(f"  Total entities : {total:,}")
    print(f"  Unique domains : {len(domain_counts):,}")
    print(f"\n  Top {top_n} domains by entity count:")
    df = pd.DataFrame(domain_counts.most_common(top_n), columns=["domain", "count"])
    df["% of total"] = (df["count"] / total * 100).round(2)
    print(df.to_string(index=False))
    return domain_counts

cr_domains = prevalence_report(cr_entities, "ClaimReview")
na_domains = prevalence_report(na_all,      "NewsArticle")

### 5.2 Property Coverage

Of all properties *officially defined* for a class in schema.org, what fraction are actually used by at least one entity?

This measures breadth of adoption across the vocabulary — a coverage of 20% means only 1 in 5 defined properties ever appear in the wild.

In [ ]:
def coverage_report(entities: dict, target_class: str):
    defined_props = so_type_props.get(target_class, set())
    if not defined_props:
        print(f"  No properties defined in vocab for {target_class}")
        return

    used_props = set()
    for data in entities.values():
        for prop in data["props"]:
            if prop.startswith(SO_HTTP) or prop.startswith(SO_HTTPS):
                used_props.add(prop.split("/")[-1].split("#")[-1])

    used_and_defined  = used_props & defined_props
    used_not_defined  = used_props - defined_props   # inherited/invalid
    defined_not_used  = defined_props - used_props

    coverage = len(used_and_defined) / len(defined_props) * 100

    print(f"\n{'─'*60}")
    print(f"  {target_class} — Property Coverage")
    print(f"{'─'*60}")
    print(f"  Properties defined in schema.org : {len(defined_props):>5}")
    print(f"  Properties used (at least once)  : {len(used_and_defined):>5}  ({coverage:.1f}% coverage)")
    print(f"  Properties never used            : {len(defined_not_used):>5}  ({100-coverage:.1f}% gap)")

    # Per-property usage frequency (density)
    prop_freq = Counter()
    for data in entities.values():
        for prop in data["props"]:
            if prop.startswith(SO_HTTP) or prop.startswith(SO_HTTPS):
                p = prop.split("/")[-1].split("#")[-1]
                if p in defined_props:
                    prop_freq[p] += 1
    n = len(entities) or 1

    print(f"\n  Used defined properties (freq = # entities using it, desc):")
    rows = [(p, prop_freq[p], prop_freq[p]/n*100) for p in defined_props]
    rows.sort(key=lambda x: -x[1])
    df = pd.DataFrame(rows, columns=["property", "entities_using", "density_%"])
    df["density_%"] = df["density_%"].round(2)
    print(df[df["entities_using"] > 0].to_string(index=False))
    print(f"\n  Never-used properties: {sorted(defined_not_used)}")

coverage_report(cr_entities, "ClaimReview")
coverage_report(na_all,      "NewsArticle")

### 5.3 AvP — Average Properties per Entity

AvP (Average number of filled Properties per entity) measures annotation richness. Reported raw and as a fraction of the total available properties for the class.

In [ ]:
def avp_report(entities: dict, target_class: str):
    defined_props = so_type_props.get(target_class, set())
    n_defined = len(defined_props) or 1

    prop_counts = []
    for data in entities.values():
        so_props_used = {
            prop.split("/")[-1].split("#")[-1]
            for prop in data["props"]
            if prop.startswith(SO_HTTP) or prop.startswith(SO_HTTPS)
        }
        prop_counts.append(len(so_props_used))

    if not prop_counts:
        print(f"  No entities for {target_class}")
        return

    import statistics
    avp  = statistics.mean(prop_counts)
    med  = statistics.median(prop_counts)
    std  = statistics.stdev(prop_counts) if len(prop_counts) > 1 else 0
    mn   = min(prop_counts)
    mx   = max(prop_counts)

    print(f"\n{'─'*60}")
    print(f"  {target_class} — Average Properties per Entity (AvP)")
    print(f"{'─'*60}")
    print(f"  Entities            : {len(prop_counts):>8,}")
    print(f"  Available props     : {n_defined:>8,}  (schema.org defined)")
    print(f"  AvP (mean)          : {avp:>8.2f}  ({avp/n_defined*100:.1f}% of available)")
    print(f"  Median              : {med:>8.1f}")
    print(f"  Std dev             : {std:>8.2f}")
    print(f"  Min / Max           : {mn:>4}  /  {mx:>4}")

    # Distribution histogram (text)
    buckets = Counter()
    for c in prop_counts:
        buckets[min(c, 20)] += 1   # cap at 20+

    print(f"\n  Distribution (# distinct SO properties per entity):")
    for k in range(0, min(mx + 2, 22)):
        bar_len = buckets.get(k, 0) * 40 // max(buckets.values(), default=1)
        label = f"{k:>2}" if k < 20 else "20+"
        print(f"    {label}: {'█' * bar_len} {buckets.get(k,0):,}")

avp_report(cr_entities, "ClaimReview")
avp_report(na_all,      "NewsArticle")

### 5.4 Characteristic Sets — Most Common Property Combinations

A *characteristic set* is the exact set of properties an entity uses. Finding the most frequent sets reveals the de-facto annotation patterns publishers actually follow, independent of what schema.org defines.

In [ ]:
def characteristic_sets_report(entities: dict, label: str, top_n: int = 15):
    set_counter = Counter()

    for data in entities.values():
        so_props = frozenset(
            prop.split("/")[-1].split("#")[-1]
            for prop in data["props"]
            if prop.startswith(SO_HTTP) or prop.startswith(SO_HTTPS)
        )
        set_counter[so_props] += 1

    total = len(entities) or 1
    print(f"\n{'─'*60}")
    print(f"  {label} — Characteristic Sets (top {top_n})")
    print(f"{'─'*60}")
    print(f"  Unique property sets : {len(set_counter):,}")
    print()
    for i, (prop_set, count) in enumerate(set_counter.most_common(top_n), 1):
        pct = count / total * 100
        props_str = ", ".join(sorted(prop_set)) if prop_set else "(no SO properties)"
        print(f"  #{i:>2}  [{count:>6,}  {pct:>5.1f}%]  {{{props_str}}}")
    return set_counter

cr_char_sets = characteristic_sets_report(cr_entities, "ClaimReview")
na_char_sets = characteristic_sets_report(na_all,      "NewsArticle")

## Summary

| Metric | ClaimReview | NewsArticle |
|--------|-------------|-------------|
| Source | WDC class-specific subset (merged JSON-LD+MD) | WDC JSON-LD shards (pure JSON-LD) |
| Entities | — | — |
| Namespace correctness | — | — |
| Type validity rate | — | — |
| Property validity rate | — | — |
| Range violation rate | — | — |
| Property coverage | — | — |
| AvP | — | — |

*Fill in after running all cells above.*